# CIFAR-10 Keras 3 ConvNet Walkthrough

This notebook mirrors the Keras 3 cell sequence from the CNN basics chapter. Use it for interactive learning: run one cell at a time, inspect the tensor shapes, and connect each model layer to the chapter's shape path.

For repeatable command-line runs, backend checks, smoke tests, and figure generation, use `cifar10_keras3.py` in this directory.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## 1. Select the Backend and Import Libraries

Choose the Keras backend before importing Keras. If you change `REQUESTED_BACKEND` after running this cell, restart the kernel before running the notebook again.

In [ ]:
import os
import platform
import time
from importlib.metadata import PackageNotFoundError, version


def choose_default_backend():
    if platform.system() == "Darwin" and platform.machine() == "arm64":
        try:
            import torch

            if torch.backends.mps.is_available():
                return "torch"
        except Exception:
            pass
    return "tensorflow"


# Set to "tensorflow", "torch", or "jax" before the first run if needed.
REQUESTED_BACKEND = None

if REQUESTED_BACKEND is None:
    os.environ.setdefault("KERAS_BACKEND", choose_default_backend())
else:
    os.environ["KERAS_BACKEND"] = REQUESTED_BACKEND

import keras
from keras import layers

## 2. Set Constants and Print Run Details

Keep the seed, batch size, epoch count, Python version, Keras version, backend, and device in the run record. Set `QUICK_RUN = True` for a one-epoch sanity check on a small subset.

In [ ]:
SEED = 1234
BATCH_SIZE = 128
EPOCHS = 10
VALIDATION_SIZE = 5000
QUICK_RUN = False

keras.utils.set_random_seed(SEED)


def package_version(name):
    try:
        return f"{name} {version(name)}"
    except PackageNotFoundError:
        return f"{name} not installed"


def backend_package_and_device():
    backend = keras.backend.backend()
    if backend == "torch":
        import torch

        if torch.backends.mps.is_available():
            return package_version("torch"), "torch MPS available"
        if torch.cuda.is_available():
            return package_version("torch"), "torch CUDA available"
        return package_version("torch"), "torch CPU"
    if backend == "tensorflow":
        import tensorflow as tf

        gpu_count = len(tf.config.list_physical_devices("GPU"))
        return package_version("tensorflow"), f"TensorFlow GPU devices: {gpu_count}"
    if backend == "jax":
        import jax

        jax_devices = ", ".join(str(device) for device in jax.devices())
        return package_version("jax"), "JAX devices: " + jax_devices
    return package_version(backend), "device summary unavailable"


backend_package, device_summary = backend_package_and_device()
print("Python:", platform.python_version())
print("Keras:", keras.__version__)
print("Backend:", keras.backend.backend())
print("Backend package:", backend_package)
print("Device summary:", device_summary)
print("Seed:", SEED)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Validation size:", VALIDATION_SIZE)
print("Quick run:", QUICK_RUN)

## 3. Load, Normalize, and Split CIFAR-10

Before running this cell, predict the shapes of `x_train`, `x_val`, and `x_test`. After running it, check that images are still rank-4 tensors with explicit height, width, and channel dimensions.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = (
    keras.datasets.cifar10.load_data()
)

x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train_full = y_train_full.squeeze().astype("int64")
y_test = y_test.squeeze().astype("int64")

x_train = x_train_full[:-VALIDATION_SIZE]
y_train = y_train_full[:-VALIDATION_SIZE]
x_val = x_train_full[-VALIDATION_SIZE:]
y_val = y_train_full[-VALIDATION_SIZE:]

if QUICK_RUN:
    EPOCHS = 1
    x_train = x_train[:1024]
    y_train = y_train[:1024]
    x_val = x_val[:256]
    y_val = y_val[:256]
    x_test = x_test[:256]
    y_test = y_test[:256]

print("train:", x_train.shape, y_train.shape)
print("val:", x_val.shape, y_val.shape)
print("test:", x_test.shape, y_test.shape)
print("pixel range:", float(x_train.min()), float(x_train.max()))

## 4. Define the Small ConvNet

Before running `model.summary()`, predict the shape path: `32 x 32 x 3 -> 32 x 32 x 32 -> 16 x 16 x 32 -> 16 x 16 x 64 -> 8 x 8 x 64 -> 8 x 8 x 128 -> 8192 -> 128 -> 10`.

In [ ]:
model = keras.Sequential(
    [
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10),
    ]
)

model.summary()

## 5. Compile, Train, and Validate

The final dense layer returns logits, so the sparse categorical cross-entropy loss receives `from_logits=True`. During experimentation, use validation metrics for selection and leave the test set untouched.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

start = time.perf_counter()
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
)
elapsed = time.perf_counter() - start

print(f"elapsed_seconds={elapsed:.2f}")
print(f"final_val_loss={history.history['val_loss'][-1]:.4f}")
print(f"final_val_accuracy={history.history['val_accuracy'][-1]:.4f}")

## 6. Inspect Class Predictions

Map the ten output indices back to CIFAR-10 class names. This is a plumbing check, not a full error analysis.

In [ ]:
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

logits = model.predict(x_val[:8], verbose=0)
probs = keras.ops.softmax(logits, axis=-1)
predicted = keras.ops.convert_to_numpy(keras.ops.argmax(probs, axis=-1))

print("predicted:", [class_names[int(i)] for i in predicted])
print("true:", [class_names[int(i)] for i in y_val[:8]])

## 7. Plot the Training Record with Plotnine

This optional cell gives an inline version of the training-curve artifact produced by the script. Use it only to confirm that per-epoch metrics were captured; diagnosing regularization and learning-rate behavior belongs to the neural-network training chapter.

In [ ]:
import pandas as pd
from plotnine import (
    aes,
    facet_wrap,
    geom_line,
    geom_point,
    ggplot,
    labs,
    scale_x_continuous,
    theme_minimal,
)

epochs = range(1, len(history.history["loss"]) + 1)
records = []
for panel, split, values in [
    ("Loss", "train", history.history["loss"]),
    ("Loss", "validation", history.history["val_loss"]),
    ("Accuracy", "train", history.history["accuracy"]),
    ("Accuracy", "validation", history.history["val_accuracy"]),
]:
    for epoch, value in zip(epochs, values):
        records.append({"epoch": epoch, "panel": panel, "split": split, "value": value})

data = pd.DataFrame.from_records(records)
data["panel"] = pd.Categorical(data["panel"], categories=["Loss", "Accuracy"], ordered=True)
(
    ggplot(data, aes("epoch", "value", color="split", group="split"))
    + geom_line(size=0.8)
    + geom_point(size=2.2)
    + facet_wrap("~panel", scales="free_y", nrow=1)
    + scale_x_continuous(breaks=list(epochs))
    + labs(x="epoch", y="metric value", color="split")
    + theme_minimal()
)


## 8. Final Test Evaluation for a Selected Run

Leave this cell off while tuning. Run it once only after you have selected a final model by validation evidence.

In [ ]:
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"test_loss={test_loss:.4f} test_accuracy={test_acc:.4f}")
else:
    print("test_evaluation=skipped")

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.